<a href="https://colab.research.google.com/github/Bukunmi2108/ml-journey/blob/main/research/p2/1_bit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1 Bit LLM

In [67]:
import math
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F
from transformers import AutoTokenizer

In [68]:
class BitLinear(nn.Linear):
  def forward(self, x: torch.Tensor):
    # Scale input tensor down to 8 bits
    i_scale = x.abs().max(dim=-1, keepdim=True).values.clamp(min=1e-5)
    x_quant = torch.round(x * 127 / i_scale).clamp(-128, 127)

    # Scale layer weights to ternary -,+,0
    w = self.weight
    w_scale = w.abs().mean()
    w_quant = torch.round(w / (w_scale + 1e-5)).clamp(-1, 1)
    w_ternary = w + (w_quant - w).detach()

    out_t = F.linear(x_quant, w_ternary, self.bias)

    # Scale output tensor up to 8bits
    out_t = out_t * (i_scale / 127) * (w_scale)
    return out_t

Tests for the BitLinear class

In [69]:
bit_linear_layer = BitLinear(10, 5)
x = torch.randn(1, 10)
output = bit_linear_layer(x)

assert isinstance(output, torch.Tensor), "Assertion Failed: Output should be a torch.Tensor"
assert output.shape == (1, 5), f"Assertion Failed: Output shape mismatch. Expected (1, {5}), got {output.shape}"
assert not torch.isnan(output).any(), "Assertion Failed: Output contains NaN values"
assert not torch.isinf(output).any(), "Assertion Failed: Output contains Inf values"
print(f"Output tensor (first 5 elements): {output.flatten()[:5].tolist()}")

Output tensor (first 5 elements): [-0.23739619553089142, 0.6750513911247253, -0.06967857480049133, 0.7859424352645874, 0.9358330368995667]


Replace Linear Layers with BitLinear

In [70]:
def replace_linears_with_bitlinear(model: nn.Module):
  for name, module in model.named_children():
    if isinstance(module, nn.Linear):
      has_bias = module.bias is not None
      bit_layer = BitLinear(module.in_features, module.out_features, bias=has_bias)
      setattr(model, name, bit_layer)
    else:
      replace_linears_with_bitlinear(module)

Test the replacement function

In [71]:
class TestModule(nn.Module):
  def __init__(self):
    super().__init__()
    self.l1, self.relu, self.l2 = nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 5)

  def forward(self, x):
    return self.l2(self.relu(self.l1(x)))

original_model = TestModule()

print("Original model structure:")
print(original_model)

replace_linears_with_bitlinear(original_model)

print("\nModel structure after replacement:")
print(original_model)

assert isinstance(original_model.l1, BitLinear), "l1 was not replaced by BitLinear"
assert isinstance(original_model.l2, BitLinear), "l2 was not replaced by BitLinear"

Original model structure:
TestModule(
  (l1): Linear(in_features=10, out_features=20, bias=True)
  (relu): ReLU()
  (l2): Linear(in_features=20, out_features=5, bias=True)
)

Model structure after replacement:
TestModule(
  (l1): BitLinear(in_features=10, out_features=20, bias=True)
  (relu): ReLU()
  (l2): BitLinear(in_features=20, out_features=5, bias=True)
)


Setitng up the Transformer Blocks

Tests for RMS Norm Layer

In [72]:
class RMSNorm(nn.Module):
  def __init__(self, dim: int, eps = 1e-6):
    super().__init__()
    self.eps = eps
    self.weight = nn.Parameter(torch.ones(dim))
  def forward(self, x: torch.Tensor):
    var = x.pow(2).mean(dim=-1, keepdim=True)
    return x * torch.rsqrt(var + self.eps) * self.weight

In [73]:
rms_norm_layer = RMSNorm(10)
x_rms = torch.randn(1, 10)
output_rms = rms_norm_layer(x_rms)
assert output_rms.shape == (1, 10), f"Assertion Failed: RMSNorm output shape mismatch. Expected (1, 10), got {output_rms.shape}"
assert not torch.isnan(output_rms).any(), "Assertion Failed: RMSNorm output contains NaN values"
assert not torch.isinf(output_rms).any(), "Assertion Failed: RMSNorm output contains Inf values"
print(f"RMSNorm Output tensor (first 5 elements): {output_rms.flatten()[:5].tolist()}")

RMSNorm Output tensor (first 5 elements): [1.0318037271499634, 0.7335687875747681, -0.6301392316818237, -0.9455806016921997, 1.3178902864456177]


BitNet Causal Self Attention

In [74]:
class CausalSelfAttention(nn.Module):
  def __init__(self, dim: int, n_heads: int):
    super().__init__()
    self.dim = dim
    self.n_heads = n_heads
    self.head_dim = dim // n_heads
    self.qkv_proj = BitLinear(dim, 3 * dim, bias=False)
    self.out_proj = BitLinear(dim, dim, bias=False)
  def forward(self, x):
    B, T, C = x.shape
    qkv = self.qkv_proj(x)
    q, k, v = qkv.split(self.dim, dim=-1)

    q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    mask = torch.tril(torch.ones(T, T), diagonal=0).bool()
    scores = q @ k.transpose(-1, -2) / math.sqrt(self.head_dim)
    scores = scores.masked_fill(~mask, float("-inf"))
    probs = F.softmax(scores, dim=-1)
    context = probs @ v
    context = context.transpose(1, 2).contiguous().view(B, T, C)
    return self.out_proj(context)

Tests for the CausalSelfAttention class

In [75]:
attention_layer = CausalSelfAttention(dim=64, n_heads=4)
x_attn = torch.randn(1, 10, 64)
output_attn = attention_layer(x_attn)
assert isinstance(output_attn, torch.Tensor), "Assertion Failed: CausalSelfAttention output should be a torch.Tensor"
assert output_attn.shape == (1, 10, 64), f"Assertion Failed: CausalSelfAttention output shape mismatch. Expected (1, 10, 64), got {output_attn.shape}"
assert not torch.isnan(output_attn).any(), "Assertion Failed: CausalSelfAttention output contains NaN values"
assert not torch.isinf(output_attn).any(), "Assertion Failed: CausalSelfAttention output contains Inf values"
print(f"CausalSelfAttention Output tensor shape: {output_attn.shape}")

CausalSelfAttention Output tensor shape: torch.Size([1, 10, 64])


BitNet Feed Forward Layer

In [76]:
from prompt_toolkit.lexers import base
class BitFFN(nn.Module):
  def __init__(self, dim:int, h_dim: int):
    super().__init__()
    self.gate_proj = BitLinear(dim, h_dim, bias=False)
    self.up_proj = BitLinear(dim, h_dim, bias=False)
    self.down_proj = BitLinear(h_dim, dim, bias=False)
  def forward(self, x):
    return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

Tests for the BitFFN class

In [77]:
ffn_layer = BitFFN(dim=64, h_dim=256)
x_ffn = torch.randn(1, 10, 64)
output_ffn = ffn_layer(x_ffn)
assert isinstance(output_ffn, torch.Tensor), "Assertion Failed: BitFFN output should be a torch.Tensor"
assert output_ffn.shape == (1, 10, 64), f"Assertion Failed: BitFFN output shape mismatch. Expected (1, 10, 64), got {output_ffn.shape}"
assert not torch.isnan(output_ffn).any(), "Assertion Failed: BitFFN output contains NaN values"
assert not torch.isinf(output_ffn).any(), "Assertion Failed: BitFFN output contains Inf values"
print(f"BitFFN Output tensor shape: {output_ffn.shape}")

BitFFN Output tensor shape: torch.Size([1, 10, 64])


Single Transformer Block

In [78]:
class BitBlock(nn.Module):
  def __init__(self, dim: int, n_heads: int, h_dim: int):
    super().__init__()
    self.rn1 = RMSNorm(dim)
    self.rn2 = RMSNorm(dim)
    self.attn = CausalSelfAttention(dim, n_heads)
    self.ffn = BitFFN(dim, h_dim)
  def forward(self, x):
    return x + self.attn(self.rn1(x)) + self.ffn(self.rn2(x))

Tests for the transformer block

In [79]:
bit_block = BitBlock(dim=64, n_heads=4, h_dim=256)
x_block = torch.randn(1, 10, 64)
output_block = bit_block(x_block)
assert isinstance(output_block, torch.Tensor), "Assertion Failed: BitBlock output should be a torch.Tensor"
assert output_block.shape == (1, 10, 64), f"Assertion Failed: BitBlock output shape mismatch. Expected (1, 10, 64), got {output_block.shape}"
assert not torch.isnan(output_block).any(), "Assertion Failed: BitBlock output contains NaN values"
assert not torch.isinf(output_block).any(), "Assertion Failed: BitBlock output contains Inf values"
print(f"BitBlock Output tensor shape: {output_block.shape}")

BitBlock Output tensor shape: torch.Size([1, 10, 64])


The Full BitNet Module

In [80]:
class BitNet(nn.Module):
  def __init__(self, dim: int, vocab_size: int, n_heads: int, h_dim: int, n_layers: int):
    super().__init__()
    self.emb = nn.Embedding(vocab_size, dim)
    self.blocks = nn.ModuleList([BitBlock(dim, n_heads, h_dim) for _ in range(n_layers)])
    self.final_norm = RMSNorm(dim)
    self.lm_head = nn.Linear(dim, vocab_size)
  def forward(self, x, targets=None):
    x = self.emb(x)
    for block in self.blocks:
      x = block(x)
    x = self.final_norm(x)
    logits = self.lm_head(x)
    loss = None
    if targets is not None:
      shift_logits = logits[..., :-1, :]
      shift_targets = targets[..., 1:]
      loss = F.cross_entropy(shift_logits.permute(0, 2, 1), shift_targets)
    return logits, loss

## Tests for the `BitNet` class

In [81]:
vocab_size = 100
max_seq_len = 20
dim = 64
n_heads = 4
h_dim = 256
n_layers = 2

bitnet_model = BitNet(dim=dim, vocab_size=vocab_size, n_heads=n_heads, h_dim=h_dim, n_layers=n_layers)

x_input = torch.randint(0, vocab_size, (1, max_seq_len))
targets_input = torch.randint(0, vocab_size, (1, max_seq_len))
logits, loss = bitnet_model(x_input, targets_input)

assert isinstance(logits, torch.Tensor), "Assertion Failed: BitNet logits should be a torch.Tensor"
assert logits.shape == (1, max_seq_len, vocab_size), f"Assertion Failed: BitNet logits shape mismatch. Expected (1, {max_seq_len}, {vocab_size}), got {logits.shape}"
assert not torch.isnan(logits).any(), "Assertion Failed: BitNet logits contain NaN values"
assert not torch.isinf(logits).any(), "Assertion Failed: BitNet logits contain Inf values"
assert isinstance(loss, torch.Tensor), "Assertion Failed: BitNet loss should be a torch.Tensor"
assert loss.ndim == 0, "Assertion Failed: BitNet loss should be a scalar"
assert not torch.isnan(loss).any(), "Assertion Failed: BitNet loss contains NaN values"
assert not torch.isinf(loss).any(), "Assertion Failed: BitNet loss contains Inf values"
print(f"BitNet Logits shape: {logits.shape}")
print(f"BitNet Loss: {loss.item()}")

BitNet Logits shape: torch.Size([1, 20, 100])
BitNet Loss: 4.697406768798828


Setting up Tokenizer

In [82]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/bitnet-b1.58-2B-4T")
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token
text = "Now tell me, are you working well?"
print(tokenizer.encode(text))
print(tokenizer.pad_token)

[128000, 7184, 3371, 757, 11, 527, 499, 3318, 1664, 30]
<|eot_id|>


Dynamic Tokenizer Function

In [83]:
def collate_fn(batch, max_length):
  texts = [item["text"] for item in batch]
  encodings = tokenizer(texts, truncation=True, max_length=max_length, padding="longest", return_tensors="pt")

  input_ids = encodings["input_ids"]
  labels = input_ids.clone()
  labels[labels == tokenizer.pad_token_id] = -100

  return {"input_ids": input_ids, "labels": labels}

Hyperparameters

In [84]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 16
max_length = 1024

Loading Data

In [85]:
pile_data = load_dataset("monology/pile-uncopyrighted", split="train", streaming=True)

pile_loader = DataLoader(
    pile_data,
    batch_size=batch_size,
    collate_fn=lambda batch: collate_fn(batch, max_length),
    pin_memory=True
)

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]